In [ ]:
import os
if not os.path.isdir("data/raw"):
    os.chdir("..")


In [ ]:
import os
from pathlib import Path
print("Kernel CWD:", os.getcwd())
print("Has data/raw:", os.path.isdir("data/raw"))
print("Has data/processed:", os.path.isdir("data/processed"))
fe_path = os.path.join(os.getcwd(), "data/processed/feature_engineered_train.csv")
print("Resolved path:", fe_path)
print("File exists:", os.path.isfile(fe_path))
import pandas as pd
df = pd.read_csv(fe_path, nrows=1)
print("File columns:", len(df.columns))
print("lat in file:", "lat" in df.columns)
print("NaN count:", df.isna().sum().sum())


In [ ]:
# ================================================
# 1. Imports
# ================================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [ ]:
# ================================================
# 2. Load datasets (train + eval)
# ================================================
train_df = pd.read_csv("data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("data/processed/feature_engineered_eval.csv")

In [ ]:
'''
# ================================================
# 3. Drop high VIF features (both train + eval)
# ================================================
high_vif_features = [
    "median_sale_price" #highest correlation to 'price' => data leakage
]
train_df.drop(columns=high_vif_features, inplace=True)
eval_df.drop(columns=high_vif_features, inplace=True)
'''

In [ ]:
# ================================================
# 4. Define target & features
# ================================================
target = "price"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_eval = eval_df.drop(columns=[target])
y_eval = eval_df[target]

In [ ]:
# Drop rows with NaN (lat/lng were not merged, causing NaN)
print(f"Before NaN drop: X_train={X_train.shape}, X_eval={X_eval.shape}")
train_mask = X_train.notna().all(axis=1)
eval_mask = X_eval.notna().all(axis=1)
X_train = X_train.loc[train_mask]
y_train = y_train[train_mask]
X_eval = X_eval.loc[eval_mask]
y_eval = y_eval[eval_mask]
print(f"After NaN drop: X_train={X_train.shape}, X_eval={X_eval.shape}")

# ================================================
# 5. Standardization (fit on train, transform eval)
# ================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled  = scaler.transform(X_eval)

In [ ]:
# ================================================
# 6. Train & Evaluate Models
# ================================================

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_eval_scaled)

print("Linear Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lr))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lr)))
print(" R²:", r2_score(y_eval, y_pred_lr))

In [ ]:
# --- Ridge Regression ---
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_eval_scaled)

print("\nRidge Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_ridge))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_ridge)))
print(" R²:", r2_score(y_eval, y_pred_ridge))

In [ ]:
# --- Lasso Regression ---
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_eval_scaled)

print("\nLasso Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_lasso))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_lasso)))
print(" R²:", r2_score(y_eval, y_pred_lasso))

In [ ]:
# --- ElasticNet ---
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
y_pred_elastic = elastic.predict(X_eval_scaled)

print("\nElasticNet Regression:")
print(" MAE:", mean_absolute_error(y_eval, y_pred_elastic))
print(" RMSE:", np.sqrt(mean_squared_error(y_eval, y_pred_elastic)))
print(" R²:", r2_score(y_eval, y_pred_elastic))